__Notes:__

* This ipynb scraps tweets by data, using the getOldTweets library by Jefferson Henrique at https://github.com/Jefferson-Henrique/GetOldTweets-python.

* I scrap tweets by month, storing metadata:
    * username
    * date
    * author_id
    * content (tweet message)
    * count of (any) hashtags 
    * count of times tweet was added to a favorite list
    * count of retweets
    
* output is saved to ./tweets-data/\**month*\*.csv

In [ ]:
%load_ext jupyternotify

import timeit
import pandas as pd
import numpy as np
from datetime import timedelta, date
import os

from got3 import manager

In [ ]:
# dates in yyyy-mm-dd
tc = manager.TweetCriteria().setSince("2017-10-1").setUntil("2017-10-31").setQuerySearch("#metoo")

In [ ]:
%%notify -m "completed feb2017 scrap"

start_time = timeit.default_timer()

tweets = manager.TweetManager.getTweets(tweetCriteria=tc)

print("Got {} tweets in {:.0f} min.".format((len(tweets)), (timeit.default_timer() - start_time)/60))

In [ ]:
%%notify -m "completed df parsing"

df = pd.DataFrame(columns=['username', 'date', 'author_id', 'content', 'count_hashtags', 'favorites', 'retweets'])

start_time = timeit.default_timer()

for tweet in tweets:
    ix = len(df)

    df.at[ix, 'username'] = tweet.username.split(' ')[0]
    df.at[ix, 'date'] = tweet.date
    df.at[ix, 'author_id'] = tweet.author_id
    df.at[ix, 'content'] = tweet.text
    df.at[ix, 'count_hashtags'] = tweet.hashtags.count('#')
    df.at[ix, 'favorites'] = tweet.favorites
    df.at[ix, 'retweets'] = tweet.retweets
        
df['retweets'] = df['retweets'].astype(np.uint64)
df['favorites'] = df['favorites'].astype(np.uint64)
df['count_hashtags'] = df['count_hashtags'].astype(np.uint64)
df['author_id'] = df['author_id'].astype(np.uint64)

print("Completed in {0:.1f} min.".format((timeit.default_timer() - start_time)/60))    

In [ ]:
df.head(3)

In [ ]:
dupes = df[df.duplicated(['username', 'date', 'author_id', 'content', 'count_hashtags', 'favorites', 'retweets'], keep=False)]

In [ ]:
dupes

In [ ]:
df.drop_duplicates(subset=df.columns, keep='first', inplace=True)

In [ ]:
len(df)

In [ ]:
df.to_csv('../tweets-data/pre-2018/oct2017.csv', index=False)